# NFXP — Original Monte Carlo Model (finite types)

**Oliver Ovdal Eiberg Jørgensen & Solveig Røndal-Liniger** — Dynamic Programming, Spring 2026

---

## Model

This notebook extends `nfxp_original.ipynb` with **$K=2$ permanent discrete consumer types**. At the beginning of the panel, consumer $i$ draws an unobserved type $k_i \in \{1,2\}$ with probability $\pi_k$ and remains that type in every subsequent period.

Types differ only in their relative preference for Brand 2:

$$
\alpha^k = [0, \alpha_2^k], \qquad k \in \{1,2\}
$$

The remaining structural parameters are common across types:

$$
\theta = [\alpha_2^1,\; \alpha_2^2,\; \pi_1,\; \gamma,\; \beta^{sc},\; \beta^{dep}], \qquad \pi_2 = 1 - \pi_1
$$

The consumer chooses each period $t$: $y_{it} \in \{0, 1, \ldots, J\}$ — either no purchase ($y=0$) or brand $j \in \{1,\ldots,J\}$.

**State variable:** $x_{it} = (\ell_{it},\, d_{it},\, e_t)$ where
- $\ell_{it}$: last-purchased brand
- $d_{it}$: time since last purchase
- $e_t$: promotion status (exogenous Markov chain)

Because $k_i$ is fixed and unobserved, it is not an additional observed state variable. Conditional on a type, the state transitions and dynamic decision problem are the same as in the homogeneous model.

**Utility for type $k$:**
$$
U^k_{it} = \begin{cases}
  \alpha^k(\ell_{it}) - \beta^{\rm dep}\cdot d_{it} & \text{if } y_{it}=0 \\
  \alpha^k(j) - \gamma\cdot p_{it}(j) - \beta^{\rm sc}\mathbf{1}\{\ell_{it}\neq j\} & \text{if } y_{it}=j
\end{cases}
$$

**Mixture likelihood:**
$$
\ell(\theta) = \sum_i \log\!\left(\sum_{k=1}^{2} \pi_k \prod_t P^k(y_{it}\mid x_{it};\theta)\right)
$$

Relative to the original notebook, the probability of a consumer's complete history is therefore a mixture of type-specific history probabilities. Mixing occurs after multiplying over time because the same type generates all of a consumer's decisions.

The type labels are observationally interchangeable without a normalization, so the labels are ordered by imposing $\alpha_2^1 < \alpha_2^2$.


## 1. Import

In [ ]:
# Imports used across simulation, estimation, and plots
import time                         # timing helper
import math                         # scalar math utilities
from pathlib import Path            # filesystem path helper
import numpy as np                  # array computing
import pandas as pd                 # tabular summaries
from scipy.optimize import minimize # numerical optimizer
from scipy.special import logsumexp # stable mixture log-likelihood
import matplotlib.pyplot as plt     # plotting API
import matplotlib.ticker as mticker # percent axis formatting


# 1. Simulation

## 2. Primitives and True Parameters

In [ ]:
# ── Model dimensions ──────────────────────────────────────────────────────────
J     = 2       # brands
K     = 2       # latent consumer types
T     = 52      # periods per consumer
N     = 50_000  # consumers per panel
D_MAX = 26      # max duration index
DELTA = 0.95    # discount factor

# ── Price levels ──────────────────────────────────────────────────────────────
BASE_PRICES   = np.array([11.95, 24.95])    # regular prices by brand
PROMO_DISC    = np.array([1.95,  15.67])    # promo discounts by brand
PROMO_ENTRY   = np.array([10 / 52, 6 / 52]) # weekly promo start prob.
PROMO_PERSIST = 0.0                         # promo persistence prob.

# ── True structural parameters (finite-type DGP) ─────────────────────────────
# u_k(0) = α_k(ℓ) − β^dep*d; u_k(j) = α_k(j) − γ*p_j − β^sc*1{ℓ≠j}
ALPHA_2_TRUE  = np.array([0.20, 0.80])       # type-specific brand 2 utility
PI_TRUE       = np.array([0.45, 0.55])       # population type shares
GAMMA_TRUE    = 0.05                         # price disutility
BETA_SC_TRUE  = 0.40                         # switching cost
BETA_DEP_TRUE = 0.275                        # depletion rate

# ── Monte Carlo ───────────────────────────────────────────────────────────────
MC_REPS = 10                                  # MC replications
MC_SEED = 2024                                # master random seed
BURN_IN = 104                                 # discarded pre-sample periods
USE_COMMON_PROMOTIONS = True                  # shared market promo path
N_STARTS = 5                                  # pilot multi-start count
MC_N_STARTS = 3                               # MC multi-start count
START_SCALE = np.array([0.30, 0.30, 0.08, 0.02, 0.30, 0.08]) # start noise sd

# ── Choice indexing ───────────────────────────────────────────────────────────
# choice c = 0 means no purchase; c = j means buy brand j.
N_CHOICES = J + 1                             # choices: 0, 1, 2

# ── Parameter vector ──────────────────────────────────────────────────────────
PARAM_NAMES = ["alpha_2_1", "alpha_2_2", "pi_1", "gamma", "beta_sc", "beta_dep"] # theta labels
THETA_TRUE = np.array([                       # true theta vector
    ALPHA_2_TRUE[0],                           # type 1 alpha_2
    ALPHA_2_TRUE[1],                           # type 2 alpha_2
    PI_TRUE[0],                                # type 1 share
    GAMMA_TRUE,                                # price disutility
    BETA_SC_TRUE,                              # switching cost
    BETA_DEP_TRUE,                             # depletion rate
])
DEFAULT_THETA0 = np.array([0.05, 0.95, 0.50, 0.04, 0.25, 0.20]) # default start
PARAM_BOUNDS = [                               # optimizer bounds
    (-2.0, 3.0),                               # alpha_2_1
    (-2.0, 3.0),                               # alpha_2_2
    (0.02, 0.98),                              # pi_1
    (1e-4, 1.0),                               # gamma
    (0.0, 3.0),                                # beta_sc
    (0.0, 2.0),                                # beta_dep
]
ORDER_MARGIN = 1e-4                            # label-order buffer

print("True finite-type parameters:")          # header
for n, v in zip(PARAM_NAMES, THETA_TRUE):      # loop over theta entries
    print(f"  {n:<14} = {v}")                 # print true value
print(f"  pi_2          = {PI_TRUE[1]}")       # print derived type 2 share
print(f"\nK={K}  |  N_CHOICES={N_CHOICES}  |  BASE_PRICES={BASE_PRICES}  |  D_MAX={D_MAX}") # setup summary


## 3. Price Process — Hi-Lo (Assumption 2.1)

Promotion status $e_t \in \{0,1\}^J$ is a binary vector. With $J=2$ there are $2^2=4$ possible states. Brand promotions follow independent two-state Markov chains, and the joint transition probability is the product of the marginal probabilities.

In [ ]:
# Enumerate all 2^J joint promotion states as binary vectors.
promo_states = np.array(                                      # promo state table
    [[(s >> j) & 1 for j in range(J)] for s in range(2 ** J)], # binary encoding
    dtype=int                                                  # integer indicators
)
N_PROMO = len(promo_states)                                    # number of promo states


def make_promo_transition() -> np.ndarray:
    """Build the joint promotion transition matrix."""
    trans = np.empty((N_PROMO, N_PROMO))                       # transition matrix
    for s, curr in enumerate(promo_states):                    # current promo state
        prob_on = np.where(curr == 1, PROMO_PERSIST, PROMO_ENTRY) # brand on-prob.
        for sp, nxt in enumerate(promo_states):                # next promo state
            trans[s, sp] = np.prod(np.where(nxt == 1, prob_on, 1.0 - prob_on)) # joint prob.
    return trans                                               # row-stochastic matrix


PROMO_TRANS = make_promo_transition()                          # promo transition matrix
PRICE_BY_PROMO = BASE_PRICES - PROMO_DISC * promo_states       # prices by promo state


def stationary_distribution(trans: np.ndarray) -> np.ndarray:
    """Stationary distribution of a finite Markov transition matrix."""
    eigvals, eigvecs = np.linalg.eig(trans.T)                  # left eigenvectors
    idx = np.argmin(np.abs(eigvals - 1.0))                     # unit eigenvalue index
    stat = np.real(eigvecs[:, idx])                            # stationary vector
    stat = stat / stat.sum()                                   # normalize to sum one
    if np.any(stat < 0):                                       # fix sign convention
        stat = np.abs(stat)                                    # make nonnegative
        stat = stat / stat.sum()                               # renormalize
    return stat                                                # stationary probabilities


PROMO_STATIONARY = stationary_distribution(PROMO_TRANS)        # initial promo dist.


def draw_common_promo_path(n_periods: int, rng, initial_state: int = None) -> np.ndarray:
    """Draw one market-wide promotion path shared by all consumers."""
    path = np.empty(n_periods, dtype=int)                      # promo indices over time
    e_idx = int(initial_state) if initial_state is not None else int( # initial state
        rng.choice(N_PROMO, p=PROMO_STATIONARY)                # draw from stationarity
    )
    for t in range(n_periods):                                 # simulate weeks
        path[t] = e_idx                                        # store current state
        e_idx = int(rng.choice(N_PROMO, p=PROMO_TRANS[e_idx])) # draw next state
    return path                                                # common promo path


print("Unit prices by promotion state:")                       # price table header
for s, e in enumerate(promo_states):                           # loop over states
    p = PRICE_BY_PROMO[s]                                      # state-specific prices
    e_tuple = tuple(int(v) for v in e)                         # plain Python tuple
    print(f"  State {s}: e={e_tuple}  p={p}")                  # print price row


## 4. Utility Function: Permanent Heterogeneity

For a consumer of latent type $k$, deterministic flow utility is

$$
U^k_{it} = \begin{cases}
  \alpha^k(\ell_{it}) - \beta^{\rm dep} \cdot d_{it} & \text{if } y_{it}=0 \\
  \alpha^k(j) - \gamma\cdot p_{it}(j) - \beta^{\rm sc}\mathbf{1}\{\ell_{it}\neq j\} & \text{if } y_{it}=j
\end{cases}
$$

- $\alpha^k=[0,\alpha_2^k]$: only the Brand 2 utility intercept varies across types; Brand 1 remains normalized to zero
- $\gamma \cdot p_{it}(j)$: disutility from expenditure (linear in price)
- $\beta^{\rm dep} \times d$: inventory depletion/depreciation linear in duration, common across brands
- $\beta^{\rm sc}$: common brand switching cost paid only when $j \neq \ell$

The extension is persistent preference heterogeneity, not period-by-period taste shocks: a consumer with high $\alpha_2^k$ values Brand 2 more throughout the entire panel.

In [ ]:
def flow_util(choice, last_brand, duration, e_idx,
              alpha, gamma, beta_sc, beta_dep) -> float:
    """Deterministic period utility for one state-choice pair."""
    l = last_brand - 1                                         # last brand index
    prices = BASE_PRICES - PROMO_DISC * promo_states[e_idx]    # prices in state e

    if choice == 0:                                            # no-purchase option
        return alpha[l] - beta_dep[l] * duration               # inventory utility

    j = choice - 1                                             # chosen brand index
    return alpha[j] - gamma * prices[j] - beta_sc[l, j]        # purchase utility


## 5. Inner Loop: Value Function Iteration (VFI)

VFI is solved separately for each latent type, using that type's $\alpha_2^k$ and the common parameters $\gamma$, $\beta^{sc}$, and $\beta^{dep}$.

For each $k$, the dynamic logit Bellman equation is

$$
V^k(s) = \log\!\left(\sum_{a=0}^{J}\exp\left\{u^k(a,s)+\delta\,\mathbb{E}[V^k(s')\mid s,a]\right\}\right).
$$

**State space:** $(\ell, d, e)$, $J \times (D_{\max}+1) \times N_e = 2 \times 27 \times 4 = 216$ states for each type.

$V$ has shape `(J, D_MAX+1, N_PROMO)`.

Transition rules in VFI:
- No purchase: $(\ell, d_{\rm idx}) \to (\ell, \min(d_{\rm idx}+1, D_{\rm MAX}))$
- Purchase $j$: $(\ell, d_{\rm idx}) \to (j, 0)$ (paper-$d=1$)


In [ ]:
def solve_vfi(alpha,                         # type-specific brand utility
              gamma,                         # price coefficient
              beta_sc,                       # switching-cost matrix
              beta_dep,                      # depletion vector
              tol: float = 1e-10,            # VFI tolerance
              max_iter: int = 2_000) -> np.ndarray: # VFI iteration cap
    """Solve the Bellman equation for one finite-mixture type."""
    duration_idx = np.arange(D_MAX + 1)                       # duration grid index
    duration = duration_idx + 1.0                             # paper duration
    next_duration = np.minimum(duration_idx + 1, D_MAX)       # capped no-buy duration

    no_purchase_util = alpha[:, None] - beta_dep[:, None] * duration[None, :] # u0
    Q = np.empty((J, D_MAX + 1, N_PROMO, N_CHOICES))          # choice value array
    V = np.zeros((J, D_MAX + 1, N_PROMO))                     # initial value function

    for _ in range(max_iter):                                 # Bellman iterations
        EV = (V.reshape(J * (D_MAX + 1), N_PROMO) @ PROMO_TRANS.T # integrate e'
              ).reshape(J, D_MAX + 1, N_PROMO)                # restore state shape

        Q[..., 0] = no_purchase_util[:, :, None] + DELTA * EV[:, next_duration, :] # no-buy Q

        for j_idx in range(J):                                # purchase alternatives
            Q[..., j_idx + 1] = (                             # brand-j Q value
                alpha[j_idx]                                  # brand utility
                - gamma * PRICE_BY_PROMO[:, j_idx][None, None, :] # price disutility
                - beta_sc[:, j_idx][:, None, None]            # switching cost
                + DELTA * EV[j_idx, 0, :][None, None, :]      # post-purchase value
            )

        q_max = Q.max(axis=3)                                 # log-sum-exp shift
        V_new = q_max + np.log(np.exp(Q - q_max[..., None]).sum(axis=3)) # Emax

        if np.max(np.abs(V_new - V)) < tol:                   # convergence check
            return V_new                                      # converged value
        V = V_new                                             # update iterate

    return V                                                  # return last iterate


## 6. Conditional Choice Probabilities (CCPs)

Conditional on type $k$, the choice probability for alternative $a \in \{0,\ldots,J\}$ is

$$
P^k(a \mid s;\theta) = \frac{\exp(Q_a^k(s))}{\sum_{b=0}^{J}\exp(Q_b^k(s))}, \qquad Q_a^k(s) = u^k(a,s) + \delta\,\mathbb{E}[V^k(s')\mid s,a].
$$

A single type-specific `P` has shape `(J, D_MAX+1, N_PROMO, N_CHOICES)`. The estimation routine stacks these arrays into `P_by_type` with shape `(K, J, D_MAX+1, N_PROMO, N_CHOICES)`.

In [ ]:
def compute_ccps(V,                          # solved value function
                 alpha,                      # type-specific brand utility
                 gamma,                      # price coefficient
                 beta_sc,                    # switching-cost matrix
                 beta_dep) -> np.ndarray:    # depletion vector
    """Compute CCPs from a solved value function."""
    duration_idx = np.arange(D_MAX + 1)                       # duration grid index
    duration = duration_idx + 1.0                             # paper duration
    next_duration = np.minimum(duration_idx + 1, D_MAX)       # capped no-buy duration

    EV = (V.reshape(J * (D_MAX + 1), N_PROMO) @ PROMO_TRANS.T # integrate e'
          ).reshape(J, D_MAX + 1, N_PROMO)                    # restore state shape

    Q = np.empty((J, D_MAX + 1, N_PROMO, N_CHOICES))          # choice value array
    no_purchase_util = alpha[:, None] - beta_dep[:, None] * duration[None, :] # u0
    Q[..., 0] = no_purchase_util[:, :, None] + DELTA * EV[:, next_duration, :] # no-buy Q

    for j_idx in range(J):                                    # purchase alternatives
        Q[..., j_idx + 1] = (                                 # brand-j Q value
            alpha[j_idx]                                      # brand utility
            - gamma * PRICE_BY_PROMO[:, j_idx][None, None, :] # price disutility
            - beta_sc[:, j_idx][:, None, None]                # switching cost
            + DELTA * EV[j_idx, 0, :][None, None, :]          # post-purchase value
        )

    weights = np.exp(Q - Q.max(axis=3, keepdims=True))        # softmax numerator
    return weights / weights.sum(axis=3, keepdims=True)       # CCP array


## 7. Data Simulation

`simulate_panel` draws one type $k_i$ per consumer, keeps that type fixed over the complete simulated history, and then draws choices from the appropriate type-specific CCPs. It returns:
- `Y` ∈ {0,...,J}: brand choice (0 = no purchase)
- `L`: last-purchased brand (state variable)
- `D`: duration since last purchase, d_idx ∈ {0,...,D_MAX}
- `E_IDX`: promotion status
- `TYPE`: latent type used by the Monte Carlo DGP, retained only for simulation diagnostics

The DGP uses a burn-in period for initial endogenous states and one common weekly promotion path faced by all consumers. In estimation, `TYPE` is not observed by the likelihood; the estimator integrates it out using the type shares $\pi_k$.


In [ ]:
def _sample_rows(rng,                         # random generator
                 row_probs: np.ndarray) -> np.ndarray: # row-wise probabilities
    """Vectorized categorical draw: one observation per row."""
    u = rng.random(len(row_probs))                            # uniform draws
    cumsum = np.cumsum(row_probs, axis=1)                     # cumulative probs
    return (u[:, None] > cumsum).sum(axis=1)                  # chosen categories


def simulate_panel(P_by_type: np.ndarray,                     # CCPs by type
                   type_shares: np.ndarray = PI_TRUE,         # type probabilities
                   n_consumers: int = N,                      # panel size
                   n_periods: int = T,                        # observed periods
                   seed=None,                                 # RNG seed
                   common_promotions: bool = USE_COMMON_PROMOTIONS, # shared promo path
                   burn_in: int = BURN_IN,                    # pre-sample periods
                   promo_path: np.ndarray = None,             # optional promo path
                   return_promo_path: bool = False) -> dict:  # return path flag
    """Simulate a finite-type consumer panel from type-specific CCPs."""
    rng = np.random.default_rng(seed)                         # initialize RNG
    P_by_type = np.asarray(P_by_type, dtype=float)            # ensure ndarray
    if P_by_type.ndim == 4:                                   # allow single-type input
        P_by_type = P_by_type[None, ...]                      # add type axis

    n_types = P_by_type.shape[0]                              # number of types
    type_shares = np.asarray(type_shares, dtype=float)        # ensure ndarray
    type_shares = type_shares / type_shares.sum()             # normalize shares
    if len(type_shares) != n_types:                           # dimension guard
        raise ValueError("type_shares must match the first dimension of P_by_type") # bad shares

    type_ids = rng.choice(n_types, size=n_consumers, p=type_shares) # draw types
    total_periods = burn_in + n_periods                       # simulated horizon

    Y = np.zeros((n_consumers, n_periods), dtype=int)         # choices
    L = np.zeros((n_consumers, n_periods), dtype=int)         # last brands
    D = np.zeros((n_consumers, n_periods), dtype=int)         # durations
    E_IDX = np.zeros((n_consumers, n_periods), dtype=int)     # promo states

    ell = rng.integers(1, J + 1, size=n_consumers)            # initial last brand
    dur = np.zeros(n_consumers, dtype=int)                    # initial duration

    if common_promotions:                                     # shared market path
        if promo_path is None:                                # no supplied path
            promo_path = draw_common_promo_path(total_periods, rng) # draw path
        else:                                                 # validate supplied path
            promo_path = np.asarray(promo_path, dtype=int)    # ensure int array
            if len(promo_path) < total_periods:               # length guard
                raise ValueError("promo_path must cover burn_in + n_periods") # too short
    else:                                                     # individual paths
        e_idx = rng.choice(N_PROMO, size=n_consumers, p=PROMO_STATIONARY) # initial e

    for tau in range(total_periods):                          # simulate all periods
        if common_promotions:                                 # shared e_t
            e_lookup = np.full(n_consumers, promo_path[tau], dtype=int) # common state
        else:                                                 # individual e_it
            e_lookup = e_idx                                  # current states

        if tau >= burn_in:                                    # observed sample
            t = tau - burn_in                                 # sample index
            L[:, t] = ell                                     # record last brand
            D[:, t] = dur                                     # record duration
            E_IDX[:, t] = e_lookup                            # record promo state

        probs = P_by_type[type_ids, ell - 1, np.minimum(dur, D_MAX), e_lookup, :] # CCP lookup
        y = _sample_rows(rng, probs)                          # draw choices
        if tau >= burn_in:                                    # observed sample
            Y[:, t] = y                                       # record choice

        bought = y > 0                                        # purchase indicator
        ell = np.where(bought, y, ell)                        # update last brand
        dur = np.where(bought, 0, np.minimum(dur + 1, D_MAX)) # update duration
        if not common_promotions:                             # individual paths
            e_idx = _sample_rows(rng, PROMO_TRANS[e_idx])     # update promo state

    out = {"Y": Y, "L": L, "D": D, "E_IDX": E_IDX, "TYPE": type_ids} # panel dict
    if return_promo_path and common_promotions:               # optional path output
        out["PROMO_PATH"] = promo_path[burn_in:burn_in + n_periods] # sample path
    return out                                                # simulated panel


## 8. Log-Likelihood and NFXP

In the homogeneous original model, one set of CCPs gives the likelihood of every consumer history. With permanent unobserved heterogeneity, the conditional history likelihood for type $k$ is
$$
L_i^k(\theta) = \prod_t P^k(y_{it} \mid \ell_{it}, d_{it}, e_{it}; \theta).
$$

Since type is latent, the finite-mixture log-likelihood is
$$
\ell(\theta) = \sum_i \log\!\left(\sum_{k=1}^{2} \pi_k L_i^k(\theta)\right)
= \sum_i \log\!\left(\sum_{k=1}^{2} \pi_k \prod_t P^k(y_{it} \mid \ell_{it}, d_{it}, e_{it}; \theta)\right).
$$

The mixture is at the consumer-history level, not at the observation level: choices by the same consumer jointly reveal whether that consumer is more likely to be the low- or high-Brand-2-preference type.

**Parameter vector:**
$$
\theta = [\alpha_2^1,\; \alpha_2^2,\; \pi_1,\; \gamma,\; \beta^{sc},\; \beta^{dep}], \qquad \pi_2=1-\pi_1, \qquad \alpha_2^1 < \alpha_2^2.
$$

**NFXP computation:** For each trial $\theta$ in the outer bounded L-BFGS-B optimization, the inner loop solves the Bellman equation and CCPs separately for both types. The code then computes $\log L_i^k$ for each consumer and evaluates the mixture with a numerically stable log-sum-exp. Multiple starting values help diagnose local maxima and weak separation of the two types.

The ordering restriction prevents label switching: type 1 is defined as the lower-Brand-2-preference type. Identification becomes weak if $\alpha_2^1$ and $\alpha_2^2$ are nearly equal or if a fitted type share approaches zero.


In [ ]:
def log_likelihood(data: dict,                 # simulated panel
                   P: np.ndarray) -> float:     # one-type CCPs
    """Summed log-likelihood over all observations for one type."""
    Y, L, D, E = data["Y"], data["L"], data["D"], data["E_IDX"] # panel arrays
    probs = P[L - 1, D, E, Y]                                 # realized CCPs
    return float(np.sum(np.log(np.maximum(probs, 1e-300))))   # summed log prob


def consumer_log_likelihood(data: dict,        # simulated panel
                            P: np.ndarray) -> np.ndarray: # one-type CCPs
    """Log-likelihood by consumer for one type."""
    Y, L, D, E = data["Y"], data["L"], data["D"], data["E_IDX"] # panel arrays
    probs = P[L - 1, D, E, Y]                                 # realized CCPs
    return np.log(np.maximum(probs, 1e-300)).sum(axis=1)      # sum over time


def state_choice_counts(data: dict) -> np.ndarray:
    """Aggregate panel observations by state and realized choice."""
    counts = np.zeros((J, D_MAX + 1, N_PROMO, N_CHOICES), dtype=float) # count cube
    np.add.at(                                                    # in-place counts
        counts,                                                   # target array
        (                                                         # state-choice indices
            data["L"].ravel() - 1,                                # last brand index
            data["D"].ravel(),                                    # duration index
            data["E_IDX"].ravel(),                                # promo index
            data["Y"].ravel(),                                    # choice index
        ),
        1.0,                                                      # count increment
    )
    return counts                                                 # aggregated data


def make_alpha(alpha_2: float) -> np.ndarray:
    """Build alpha vector with Brand 1 normalized to zero."""
    return np.array([0.0, float(alpha_2)])                       # [alpha_1, alpha_2]


def make_beta_dep(beta_dep: float) -> np.ndarray:
    """Expand common depletion rate to brands."""
    return np.full(J, float(beta_dep))                           # brand vector


def make_beta_sc(beta_sc: float) -> np.ndarray:
    """Build symmetric switching-cost matrix."""
    mat = np.full((J, J), float(beta_sc))                        # off-diagonal cost
    np.fill_diagonal(mat, 0.0)                                   # no own-brand cost
    return mat                                                   # switching matrix


def unpack(theta: np.ndarray):
    """Unpack theta = [alpha_2_1, alpha_2_2, pi_1, gamma, beta_sc, beta_dep]."""
    theta = np.asarray(theta, dtype=float)                       # numeric theta
    alpha_2_values = theta[:K].copy()                            # type alphas
    type_shares = np.array([theta[2], 1.0 - theta[2]])           # [pi_1, pi_2]
    gamma = float(theta[3])                                      # price coefficient
    beta_sc = make_beta_sc(theta[4])                             # switching matrix
    beta_dep = make_beta_dep(theta[5])                           # depletion vector
    return alpha_2_values, type_shares, gamma, beta_sc, beta_dep # unpacked params


def invalid_theta(theta: np.ndarray) -> bool:
    """Check signs, shares, and type ordering."""
    theta = np.asarray(theta, dtype=float)                       # numeric theta
    if np.any(~np.isfinite(theta)):                              # finite guard
        return True                                              # reject invalid
    alpha_order_ok = theta[1] > theta[0] + ORDER_MARGIN          # label ordering
    pi_ok = 0.0 < theta[2] < 1.0                                 # valid share
    signs_ok = (theta[3] > 0.0) and (theta[4] >= 0.0) and (theta[5] >= 0.0) # signs
    return (not alpha_order_ok) or (not pi_ok) or (not signs_ok) # invalid flag


def solve_type_ccps(theta: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Solve VFI and CCPs for each latent type."""
    alpha_2_values, type_shares, gamma, beta_sc, beta_dep = unpack(theta) # params
    P_by_type = np.empty((K, J, D_MAX + 1, N_PROMO, N_CHOICES)) # type CCPs
    for k, alpha_2 in enumerate(alpha_2_values):               # loop over types
        alpha_k = make_alpha(alpha_2)                          # type alpha vector
        V_k = solve_vfi(alpha_k, gamma, beta_sc, beta_dep)     # type value function
        P_by_type[k] = compute_ccps(V_k, alpha_k, gamma, beta_sc, beta_dep) # type CCPs
    return P_by_type, type_shares                              # CCPs and weights


def mixture_log_likelihood(data: dict, theta: np.ndarray) -> float:
    """Household-level finite-mixture log-likelihood."""
    if invalid_theta(theta):                                   # parameter guard
        return -np.inf                                         # impossible theta
    P_by_type, type_shares = solve_type_ccps(theta)            # type CCPs
    log_pi = np.log(type_shares)                               # log type shares
    consumer_ll = np.empty((K, data["Y"].shape[0]))             # type-consumer LLs
    for k in range(K):                                         # loop over types
        consumer_ll[k] = consumer_log_likelihood(data, P_by_type[k]) # LL_i|k
    return float(logsumexp(log_pi[:, None] + consumer_ll, axis=0).sum()) # mixture LL


def nfxp_objective(theta: np.ndarray, data: dict) -> float:
    """Return negative mixture log-likelihood for the optimizer."""
    theta = np.asarray(theta, dtype=float)                     # numeric theta
    if invalid_theta(theta):                                   # constraint guard
        order_violation = max(theta[0] + ORDER_MARGIN - theta[1], 0.0) # alpha order
        sign_violation = np.minimum(theta[3:], 0.0)            # sign violations
        pi_violation = max(0.0 - theta[2], theta[2] - 1.0, 0.0) # share violation
        penalty = order_violation ** 2 + pi_violation ** 2 + float(np.sum(sign_violation ** 2)) # penalty
        return 1e12 + 1e8 * penalty                            # large objective
    return -mixture_log_likelihood(data, theta)                # negative LL


def coerce_theta(theta: np.ndarray) -> np.ndarray:
    """Project a starting value into the admissible ordered region."""
    theta = np.asarray(theta, dtype=float).copy()              # mutable copy
    theta[:2] = np.sort(theta[:2])                             # order type labels
    for i, (lo, hi) in enumerate(PARAM_BOUNDS):                # loop over bounds
        theta[i] = np.clip(theta[i], lo + 1e-5, hi - 1e-5)     # interior clip

    if theta[1] <= theta[0] + ORDER_MARGIN:                    # too-close alphas
        lo = PARAM_BOUNDS[0][0] + 1e-5                         # lower alpha bound
        hi = PARAM_BOUNDS[1][1] - 1e-5                         # upper alpha bound
        midpoint = np.clip(0.5 * (theta[0] + theta[1]), lo + 0.25, hi - 0.25) # center
        theta[0] = midpoint - 0.25                             # type 1 alpha
        theta[1] = midpoint + 0.25                             # type 2 alpha
    return theta                                               # feasible theta


def estimate_nfxp(data: dict,                         # simulated panel
                  theta0: np.ndarray = None,           # starting theta
                  verbose: bool = False):              # kept for interface
    """Estimate the finite-type model with bounded L-BFGS-B."""
    if theta0 is None:                                          # default start
        theta0 = DEFAULT_THETA0.copy()                          # copy default
    theta0 = coerce_theta(theta0)                               # feasible start
    return minimize(                                            # outer optimizer
        fun=nfxp_objective,                                     # objective
        x0=theta0,                                              # starting point
        args=(data,),                                           # fixed data
        method="L-BFGS-B",                                      # bounded method
        bounds=PARAM_BOUNDS,                                    # parameter bounds
        options={"maxiter": 500, "ftol": 1e-8, "gtol": 1e-5}, # tolerances
    )


def make_starting_values(theta_center: np.ndarray = None,       # center point
                         n_starts: int = N_STARTS,             # number of starts
                         seed=None) -> list[np.ndarray]:       # RNG seed
    """Create dispersed but admissible finite-mixture starts."""
    rng = np.random.default_rng(seed)                           # initialize RNG
    center = DEFAULT_THETA0.copy() if theta_center is None else np.asarray(theta_center, dtype=float) # center
    starts = [coerce_theta(center)]                             # include center
    for _ in range(max(n_starts - 1, 0)):                       # extra starts
        draw = center + rng.normal(0.0, START_SCALE, size=len(center)) # random draw
        starts.append(coerce_theta(draw))                       # store feasible draw
    return starts                                               # start list


def estimate_nfxp_multistart(data: dict,                       # simulated panel
                             theta_center: np.ndarray = None,  # start center
                             n_starts: int = N_STARTS,         # start count
                             seed=None,                        # RNG seed
                             verbose: bool = False):           # optimizer verbosity
    """Run NFXP from several starts and keep the best result."""
    results = []                                                # optimizer results
    for start_id, theta0 in enumerate(make_starting_values(theta_center, n_starts, seed), start=1): # starts
        res = estimate_nfxp(data, theta0=theta0, verbose=verbose) # estimate
        res.start_id = start_id                                 # record start id
        res.theta0 = theta0                                     # record start theta
        results.append(res)                                     # store result

    successful = [r for r in results if r.success]              # converged runs
    candidates = successful if successful else results          # fallback if none
    best = min(candidates, key=lambda r: r.fun)                 # lowest objective
    return best, results                                        # best and all runs


def likelihood_profile(data: dict,                              # simulated panel
                       center_theta: np.ndarray,                # reference theta
                       param_idx: int,                          # profiled index
                       grid: np.ndarray) -> pd.DataFrame:       # grid values
    """One-dimensional likelihood profile."""
    base_nll = nfxp_objective(center_theta, data)               # reference NLL
    rows = []                                                   # profile rows
    for value in grid:                                          # grid loop
        theta = center_theta.copy()                             # trial theta
        theta[param_idx] = value                                # vary parameter
        nll = nfxp_objective(theta, data)                       # trial NLL
        rows.append({                                           # store result
            "param": PARAM_NAMES[param_idx],                   # parameter name
            "value": value,                                    # grid value
            "delta_nll": nll - base_nll,                       # relative NLL
        })
    return pd.DataFrame(rows)                                   # profile table


def profile_all_parameters(data: dict,                          # simulated panel
                           center_theta: np.ndarray,            # reference theta
                           rel_width: float = 0.40,             # grid width share
                           n_grid: int = 9) -> pd.DataFrame:    # grid size
    """Likelihood profiles around a reference estimate."""
    frames = []                                                 # profile tables
    for idx, value in enumerate(center_theta):                  # parameter loop
        width = rel_width * max(abs(value), 0.05)               # local window
        lo_bound, hi_bound = PARAM_BOUNDS[idx]                  # parameter bounds
        lo = max(lo_bound + 1e-5, value - width)                # lower grid edge
        hi = min(hi_bound - 1e-5, value + width)                # upper grid edge
        if idx == 0:                                            # alpha_2_1 case
            hi = min(hi, center_theta[1] - ORDER_MARGIN)        # keep ordering
        elif idx == 1:                                          # alpha_2_2 case
            lo = max(lo, center_theta[0] + ORDER_MARGIN)        # keep ordering
        if lo >= hi:                                            # empty grid guard
            continue                                            # skip parameter
        grid = np.linspace(lo, hi, n_grid)                      # profile grid
        frames.append(likelihood_profile(data, center_theta, idx, grid)) # profile
    return pd.concat(frames, ignore_index=True)                 # stacked profiles


def numerical_hessian(func,                         # scalar objective
                      theta: np.ndarray,            # evaluation point
                      step_scale: float = 1e-4,     # relative step size
                      min_step: float = 1e-5) -> np.ndarray: # minimum step
    """Central-difference Hessian for a scalar objective."""
    theta = np.asarray(theta, dtype=float)                       # numeric theta
    k = len(theta)                                               # parameter count
    h = np.maximum(step_scale * np.maximum(np.abs(theta), 1.0), min_step) # steps
    H = np.empty((k, k), dtype=float)                            # Hessian array
    f0 = func(theta)                                             # center value

    for i in range(k):                                           # diagonal terms
        ei = np.zeros(k)                                         # unit step vector
        ei[i] = h[i]                                             # step i
        H[i, i] = (func(theta + ei) - 2.0 * f0 + func(theta - ei)) / (h[i] ** 2) # second diff

    for i in range(k):                                           # row index
        for j in range(i + 1, k):                                # column index
            ei = np.zeros(k)                                     # i-step vector
            ej = np.zeros(k)                                     # j-step vector
            ei[i] = h[i]                                         # step i
            ej[j] = h[j]                                         # step j
            H_ij = (                                             # cross derivative
                func(theta + ei + ej)                            # plus-plus
                - func(theta + ei - ej)                          # plus-minus
                - func(theta - ei + ej)                          # minus-plus
                + func(theta - ei - ej)                          # minus-minus
            ) / (4.0 * h[i] * h[j])                              # scale factor
            H[i, j] = H_ij                                       # upper triangle
            H[j, i] = H_ij                                       # lower triangle

    return H                                                     # Hessian matrix


def estimate_standard_errors(data: dict,                         # simulated panel
                             theta_hat: np.ndarray,              # MLE estimate
                             step_scale: float = 1e-4) -> tuple[pd.DataFrame, np.ndarray]: # step scale
    """Approximate MLE standard errors from the Hessian."""
    objective = lambda th: nfxp_objective(th, data)              # fixed-data NLL
    H = numerical_hessian(objective, theta_hat, step_scale=step_scale) # Hessian
    H = 0.5 * (H + H.T)                                          # symmetrize

    try:                                                         # regular inverse
        cov = np.linalg.inv(H)                                   # covariance
        covariance_method = "inverse Hessian"                    # method label
    except np.linalg.LinAlgError:                                # singular Hessian
        cov = np.linalg.pinv(H)                                  # fallback covariance
        covariance_method = "pseudo-inverse Hessian"             # method label

    diag = np.diag(cov)                                          # variances
    se = np.full_like(diag, np.nan, dtype=float)                 # standard errors
    positive_diag = diag > 0                                     # valid variances
    se[positive_diag] = np.sqrt(diag[positive_diag])             # standard errors
    z = theta_hat / se                                           # z statistics
    ci_lo = theta_hat - 1.96 * se                                # lower CI
    ci_hi = theta_hat + 1.96 * se                                # upper CI
    eig_min = float(np.linalg.eigvalsh(H).min())                 # Hessian diagnostic

    se_table = pd.DataFrame({                                    # SE summary table
        "param": PARAM_NAMES,                                   # parameter names
        "estimate": theta_hat,                                  # estimates
        "std_error": se,                                        # standard errors
        "z_stat": z,                                            # z stats
        "ci95_lo": ci_lo,                                       # lower CI
        "ci95_hi": ci_hi,                                       # upper CI
    })
    se_table.attrs["covariance_method"] = covariance_method      # store method
    se_table.attrs["hessian_min_eigenvalue"] = eig_min           # store diagnostic
    return se_table, H                                           # table and Hessian


def mixture_ccps(theta: np.ndarray) -> np.ndarray:
    """Population-weighted CCPs from the finite-type model."""
    P_by_type, type_shares = solve_type_ccps(theta)              # type CCPs
    return np.tensordot(type_shares, P_by_type, axes=(0, 0))     # weighted CCPs


def model_fit_tables(data: dict,                                 # simulated panel
                     theta_hat: np.ndarray,                      # fitted theta
                     duration_tail_start: int = 8) -> dict[str, pd.DataFrame]: # tail bin
    """Observed-vs-predicted fit summaries."""
    counts = state_choice_counts(data)                           # state-choice counts
    P_hat = mixture_ccps(theta_hat)                               # fitted CCPs
    state_counts = counts.sum(axis=3)                             # state counts

    duration_rows = []                                            # duration fit rows
    duration_bins = [(d, d, str(d + 1)) for d in range(duration_tail_start - 1)] # bins
    if duration_tail_start <= D_MAX + 1:                          # tail-bin guard
        duration_bins.append((duration_tail_start - 1, D_MAX, f"{duration_tail_start}+")) # tail bin

    for d_lo, d_hi, label in duration_bins:                       # duration bins
        denom = state_counts[:, d_lo:d_hi + 1, :].sum()           # bin observations
        if denom <= 0:                                            # empty-bin guard
            continue                                              # skip bin
        obs_no = counts[:, d_lo:d_hi + 1, :, 0].sum() / denom     # observed no-buy
        pred_no = (                                               # predicted no-buy
            state_counts[:, d_lo:d_hi + 1, :] * P_hat[:, d_lo:d_hi + 1, :, 0]
        ).sum() / denom
        obs_purchase = counts[:, d_lo:d_hi + 1, :, 1:].sum() / denom # observed buy
        pred_purchase = (                                         # predicted buy
            state_counts[:, d_lo:d_hi + 1, :] * P_hat[:, d_lo:d_hi + 1, :, 1:].sum(axis=3)
        ).sum() / denom
        duration_rows.append({                                    # append bin row
            "duration": label,                                   # duration label
            "duration_mid": 0.5 * (d_lo + d_hi) + 1,             # midpoint
            "n_obs": denom,                                      # observations
            "obs_no_purchase": obs_no,                           # observed no-buy
            "pred_no_purchase": pred_no,                         # predicted no-buy
            "obs_purchase_hazard": obs_purchase,                 # observed buy rate
            "pred_purchase_hazard": pred_purchase,               # predicted buy rate
        })
    duration_fit = pd.DataFrame(duration_rows)                    # duration table

    promo_rows = []                                               # promo fit rows
    choice_labels = ["no_purchase"] + [f"brand_{j}" for j in range(1, J + 1)] # choices
    for e_idx, e_vec in enumerate(promo_states):                  # promo states
        denom = state_counts[:, :, e_idx].sum()                   # state observations
        if denom <= 0:                                            # empty-state guard
            continue                                              # skip state
        for c, label in enumerate(choice_labels):                 # choice loop
            obs_share = counts[:, :, e_idx, c].sum() / denom      # observed share
            pred_share = (state_counts[:, :, e_idx] * P_hat[:, :, e_idx, c]).sum() / denom # predicted
            promo_rows.append({                                   # append row
                "promo_state": str(tuple(int(v) for v in e_vec)), # promo label
                "choice": label,                                 # choice label
                "n_obs": denom,                                  # observations
                "obs_share": obs_share,                          # observed share
                "pred_share": pred_share,                        # predicted share
            })
    promo_choice_fit = pd.DataFrame(promo_rows)                   # promo table

    obs_switch_all = 0.0                                          # observed switches
    pred_switch_all = 0.0                                         # predicted switches
    obs_purchase = counts[..., 1:].sum()                          # observed buys
    pred_purchase = (state_counts * P_hat[..., 1:].sum(axis=3)).sum() # predicted buys
    total_obs = counts.sum()                                      # total observations
    for l_idx in range(J):                                        # last-brand loop
        for j_idx in range(J):                                    # chosen-brand loop
            if j_idx == l_idx:                                    # no switch
                continue                                          # skip own brand
            choice_idx = j_idx + 1                                # choice index
            obs_switch_all += counts[l_idx, :, :, choice_idx].sum() # observed switch
            pred_switch_all += (state_counts[l_idx, :, :] * P_hat[l_idx, :, :, choice_idx]).sum() # predicted

    switching_fit = pd.DataFrame([                                # switching table
        {
            "metric": "purchase_rate",                          # purchase rate
            "observed": obs_purchase / total_obs,                 # observed rate
            "predicted": pred_purchase / total_obs,               # predicted rate
        },
        {
            "metric": "switch_rate_all_observations",            # switch rate
            "observed": obs_switch_all / total_obs,               # observed rate
            "predicted": pred_switch_all / total_obs,             # predicted rate
        },
        {
            "metric": "switch_rate_given_purchase",              # conditional switch
            "observed": obs_switch_all / obs_purchase if obs_purchase > 0 else np.nan, # observed
            "predicted": pred_switch_all / pred_purchase if pred_purchase > 0 else np.nan, # predicted
        },
    ])

    return {                                                       # fit outputs
        "duration_fit": duration_fit,                             # duration fit
        "promo_choice_fit": promo_choice_fit,                     # promo fit
        "switching_fit": switching_fit,                           # switch fit
    }


def plot_model_fit(fit_tables: dict[str, pd.DataFrame],           # fit tables
                   filename: str = "nfxp_model_fit_original_finite_types.pdf"): # output file
    """Plot observed-vs-predicted fit diagnostics."""
    duration_fit = fit_tables["duration_fit"]                     # duration table
    promo_choice_fit = fit_tables["promo_choice_fit"]             # promo table
    switching_fit = fit_tables["switching_fit"]                   # switching table

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))               # plot grid

    ax = axes[0, 0]                                               # no-buy panel
    x_duration = np.arange(len(duration_fit))                     # duration x-axis
    duration_labels = duration_fit["duration"].astype(str).tolist() # x labels
    ax.plot(x_duration, duration_fit["obs_no_purchase"], marker="o", label="Observed") # observed
    ax.plot(x_duration, duration_fit["pred_no_purchase"], marker="s", label="Predicted") # predicted
    ax.set_xticks(x_duration)                                     # tick positions
    ax.set_xticklabels(duration_labels)                           # tick labels
    ax.set_title("No-purchase share by duration")                 # panel title
    ax.set_xlabel("Duration since last purchase")                 # x label
    ax.set_ylabel("Share")                                        # y label
    ax.legend()                                                   # legend

    ax = axes[0, 1]                                               # purchase panel
    ax.plot(x_duration, duration_fit["obs_purchase_hazard"], marker="o", label="Observed") # observed
    ax.plot(x_duration, duration_fit["pred_purchase_hazard"], marker="s", label="Predicted") # predicted
    ax.set_xticks(x_duration)                                     # tick positions
    ax.set_xticklabels(duration_labels)                           # tick labels
    ax.set_title("Purchase hazard by duration")                   # panel title
    ax.set_xlabel("Duration since last purchase")                 # x label
    ax.set_ylabel("Share")                                        # y label
    ax.legend()                                                   # legend

    ax = axes[1, 0]                                               # promo panel
    promo_brand = promo_choice_fit[promo_choice_fit["choice"].isin(["brand_1", "brand_2"])] # brand rows
    x_labels = promo_brand["promo_state"].drop_duplicates().tolist() # promo labels
    x = np.arange(len(x_labels))                                  # x positions
    width = 0.18                                                  # bar width
    for offset, choice in [(-1.5 * width, "brand_1"), (-0.5 * width, "brand_2")]: # observed bars
        g = promo_brand[promo_brand["choice"] == choice].set_index("promo_state").loc[x_labels] # data
        ax.bar(x + offset, g["obs_share"], width, label=f"Obs {choice}", alpha=0.75) # bars
    for offset, choice in [(0.5 * width, "brand_1"), (1.5 * width, "brand_2")]: # predicted bars
        g = promo_brand[promo_brand["choice"] == choice].set_index("promo_state").loc[x_labels] # data
        ax.bar(x + offset, g["pred_share"], width, label=f"Pred {choice}", alpha=0.75) # bars
    ax.set_xticks(x)                                             # tick positions
    ax.set_xticklabels(x_labels)                                 # tick labels
    ax.set_title("Brand choice shares by promotion state")       # panel title
    ax.set_xlabel("Promotion state (brand 1, brand 2)")          # x label
    ax.set_ylabel("Share")                                       # y label
    ax.legend(fontsize=8, ncols=2)                                # legend

    ax = axes[1, 1]                                               # switching panel
    switch_plot = switching_fit[switching_fit["metric"].isin([     # plotted metrics
        "switch_rate_all_observations", "switch_rate_given_purchase"
    ])]
    labels = ["All obs.", "Given purchase"]                       # x labels
    x = np.arange(len(labels))                                    # x positions
    ax.bar(x - 0.18, switch_plot["observed"], 0.36, label="Observed", alpha=0.8) # observed
    ax.bar(x + 0.18, switch_plot["predicted"], 0.36, label="Predicted", alpha=0.8) # predicted
    ax.set_xticks(x)                                             # tick positions
    ax.set_xticklabels(labels)                                   # tick labels
    ax.set_title("Switching rates")                              # panel title
    ax.set_ylabel("Rate")                                        # y label
    ax.legend()                                                  # legend

    for ax in axes.flat:                                         # all panels
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0)) # percent axis

    fig.suptitle("Observed vs predicted model fit — finite types", fontsize=12) # title
    plt.tight_layout()                                           # compact layout
    plt.savefig(filename, bbox_inches="tight")                   # save figure
    plt.show()                                                   # display figure
    return fig                                                   # figure handle


## 9. Pilot — Single Panel

Verify that both type-specific value functions converge, that the simulated type shares and choices look reasonable, and that finite-type NFXP recovers the structural parameters from a single panel. `TYPE` is reported here only because it is known in the Monte Carlo DGP; it is integrated out in estimation.

In [ ]:
print("Step 1: Solving type-specific DPs at true parameters...") # status
P_true_by_type, pi_true = solve_type_ccps(THETA_TRUE)            # true CCPs
print("  VFI converged for both types.")                         # status

print(f"\nStep 2: Simulating finite-type panel (N={N}, T={T}, burn-in={BURN_IN})...") # status
data_pilot = simulate_panel(P_true_by_type, type_shares=pi_true, seed=42, return_promo_path=True) # pilot panel
Y = data_pilot["Y"]                                             # pilot choices
print(f"  No purchase: {(Y == 0).mean():.1%}")                  # no-buy share
for j in range(1, J + 1):                                       # brand loop
    print(f"  Brand {j}:     {(Y == j).mean():.1%}")            # brand share

type_counts = np.bincount(data_pilot["TYPE"], minlength=K) / len(data_pilot["TYPE"]) # type shares
promo_counts = np.bincount(data_pilot["E_IDX"].ravel(), minlength=N_PROMO) / data_pilot["E_IDX"].size # promo shares
print(f"  Type shares: {np.round(type_counts, 3)}")             # print type shares
print(f"  Promotion-state shares: {np.round(promo_counts, 3)}") # print promo shares

print("\nStep 3: Estimating finite-type NFXP with multi-start...") # status
t0 = time.perf_counter()                                        # start timer
res_pilot, pilot_starts = estimate_nfxp_multistart(             # pilot estimate
    data_pilot, theta_center=DEFAULT_THETA0, n_starts=N_STARTS, seed=4242 # settings
)
t1 = time.perf_counter() - t0                                   # elapsed seconds
print(                                                          # optimizer summary
    f"  Time: {t1:.1f}s  |  Best converged: {res_pilot.success}  "
    f"|  Best start: {res_pilot.start_id}/{N_STARTS}  |  Iterations: {res_pilot.nit}"
)
print(f"  Successful starts: {sum(r.success for r in pilot_starts)}/{len(pilot_starts)}") # start success

print("\n" + "─" * 65)                                          # table divider
df_pilot = pd.DataFrame({                                      # estimate table
    "Parameter": PARAM_NAMES,                                  # parameter labels
    "True": THETA_TRUE,                                        # true values
    "NFXP": res_pilot.x,                                       # estimates
    "Bias": res_pilot.x - THETA_TRUE,                          # estimate minus true
})
print(df_pilot.to_string(index=False, float_format=lambda x: f"{x:+.4f}")) # print table
print(f"  pi_2 true={1 - THETA_TRUE[2]:.4f}, estimated={1 - res_pilot.x[2]:.4f}") # pi2

profiles_pilot = profile_all_parameters(data_pilot, res_pilot.x, rel_width=0.30, n_grid=7) # LL profiles
profile_width = profiles_pilot.groupby("param")["delta_nll"].max().reset_index(name="max_delta_nll") # width
print("\nLikelihood profile width around the pilot estimate:")  # profile header
print(profile_width.to_string(index=False, float_format=lambda x: f"{x:.2f}")) # profile table

print("\nApproximate standard errors from numerical Hessian:")  # SE header
se_pilot, hessian_pilot = estimate_standard_errors(data_pilot, res_pilot.x) # SEs
print(f"  Covariance method: {se_pilot.attrs['covariance_method']}") # method
print(f"  Hessian min eigenvalue: {se_pilot.attrs['hessian_min_eigenvalue']:.4e}") # diagnostic
print(se_pilot.to_string(index=False, float_format=lambda x: f"{x:.4f}")) # SE table

print("\nObserved vs predicted fit: duration profiles")          # fit header
fit_pilot = model_fit_tables(data_pilot, res_pilot.x)           # fit tables
duration_cols = ["duration", "n_obs", "obs_no_purchase", "pred_no_purchase", # columns
                 "obs_purchase_hazard", "pred_purchase_hazard"] # columns
print(fit_pilot["duration_fit"][duration_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}")) # print

print("\nObserved vs predicted fit: choice shares by promotion state") # fit header
print(fit_pilot["promo_choice_fit"].to_string(index=False, float_format=lambda x: f"{x:.4f}")) # print

print("\nObserved vs predicted fit: purchase and switching rates") # fit header
print(fit_pilot["switching_fit"].to_string(index=False, float_format=lambda x: f"{x:.4f}")) # print

_ = plot_model_fit(fit_pilot)                                  # plot diagnostics


## 10. Monte Carlo

For each replication:
1. Draw one market-wide weekly promotion path and assign each consumer one of the two latent types
2. Simulate a new panel after burn-in using type-specific CCPs
3. Estimate the finite-mixture model with NFXP from several dispersed starting values
4. Record estimates, bias, RMSE, convergence, and multi-start diagnostics

**Expected result:** the six finite-type parameters should recover reasonably well under the correctly specified DGP. Large bias, fitted shares near their bounds, nearly equal type intercepts, or flat likelihood profiles indicate weak identification.


In [ ]:
def run_monte_carlo(n_reps: int = MC_REPS,                    # replications
                    seed: int = MC_SEED,                      # master seed
                    n_starts: int = MC_N_STARTS,              # starts per rep
                    common_promotions: bool = USE_COMMON_PROMOTIONS, # promo mode
                    burn_in: int = BURN_IN):                  # burn-in periods
    rng_master = np.random.default_rng(seed)                   # master RNG
    rep_seeds = rng_master.integers(0, 1_000_000, size=n_reps) # rep seeds

    P_t, pi_t = solve_type_ccps(THETA_TRUE)                    # true DGP CCPs

    print(f"MC  |  K={K}, J={J}, T={T}, N={N}, D_MAX={D_MAX}, reps={n_reps}, starts={n_starts}") # header
    print(                                                     # DGP summary
        "DGP: "
        f"alpha_2=({ALPHA_2_TRUE[0]:.3f}, {ALPHA_2_TRUE[1]:.3f}), "
        f"pi=({PI_TRUE[0]:.3f}, {PI_TRUE[1]:.3f}), "
        f"gamma={GAMMA_TRUE:.3f}, beta_sc={BETA_SC_TRUE:.3f}, beta_dep={BETA_DEP_TRUE:.3f}"
    )
    print(f"Simulation: common_promotions={common_promotions}, burn_in={burn_in}\n") # sim setup

    rows = []                                                  # MC result rows
    for rep in range(1, n_reps + 1):                           # replication loop
        data = simulate_panel(                                 # simulate panel
            P_t,                                                # true CCPs
            type_shares=pi_t,                                  # true shares
            n_consumers=N,                                     # panel size
            n_periods=T,                                       # periods
            seed=int(rep_seeds[rep - 1]),                      # rep seed
            common_promotions=common_promotions,               # promo mode
            burn_in=burn_in,                                   # burn-in
        )

        t0 = time.perf_counter()                               # start timer
        r, starts = estimate_nfxp_multistart(                  # estimate model
            data,                                               # simulated data
            theta_center=DEFAULT_THETA0,                       # start center
            n_starts=n_starts,                                 # start count
            seed=int(rep_seeds[rep - 1]) + 999,                # start seed
        )
        t1 = time.perf_counter() - t0                          # elapsed seconds

        n_success = sum(s.success for s in starts)             # successful starts
        fun_values = np.array([s.fun for s in starts])         # objective values
        local_gap = np.partition(fun_values, 1)[1] - fun_values.min() if len(fun_values) > 1 else np.nan # local gap
        print(                                                 # rep status
            f"  Rep {rep:>3}/{n_reps}  conv={r.success}  "
            f"success_starts={n_success}/{len(starts)}  gap={local_gap:>7.2f}  ({t1:>4.0f}s)"
        )

        for k, name in enumerate(PARAM_NAMES):                 # parameter loop
            rows.append({                                      # store estimate
                "rep": rep,                                   # replication id
                "param": name,                                # parameter name
                "true": THETA_TRUE[k],                        # true value
                "estimate": r.x[k],                           # estimate
                "bias": r.x[k] - THETA_TRUE[k],               # bias
                "sq_error": (r.x[k] - THETA_TRUE[k]) ** 2,    # squared error
                "abs_bias": abs(r.x[k] - THETA_TRUE[k]),      # absolute bias
                "converged": int(r.success),                  # convergence flag
                "best_start": r.start_id,                     # best start id
                "n_success_starts": n_success,                # successful starts
                "local_gap": local_gap,                       # best-vs-second gap
            })

    df = pd.DataFrame(rows)                                    # long MC table
    summary_rows = []                                          # summary rows
    for i, name in enumerate(PARAM_NAMES):                     # parameter loop
        g = df[df["param"] == name]                            # parameter rows
        estimates = g["estimate"].dropna()                     # valid estimates
        n_rep = int(estimates.size)                            # valid reps
        mean_est = estimates.mean()                            # mean estimate
        bias = mean_est - THETA_TRUE[i]                        # mean bias
        std = estimates.std(ddof=1)                            # estimate sd
        se_mean = std / np.sqrt(n_rep) if n_rep > 0 else np.nan # SE of mean
        rmse = np.sqrt(g["sq_error"].mean())                   # RMSE
        mean_abs_bias = g["abs_bias"].mean()                   # mean abs bias
        true_value = THETA_TRUE[i]                             # true value
        relative_bias_pct = 100 * bias / true_value if not np.isclose(true_value, 0.0) else np.nan # rel bias

        if n_rep > 1 and np.isfinite(se_mean) and se_mean > 0: # CI guard
            z_crit = 1.96                                      # normal 95% critical
            ci95_lo = mean_est - z_crit * se_mean              # lower CI
            ci95_hi = mean_est + z_crit * se_mean              # upper CI
        else:                                                  # insufficient reps
            ci95_lo = np.nan                                   # lower CI missing
            ci95_hi = np.nan                                   # upper CI missing

        summary_rows.append({                                  # store summary
            "param": name,                                    # parameter name
            "true": THETA_TRUE[i],                            # true value
            "n_rep": n_rep,                                   # valid reps
            "mean_est": mean_est,                             # mean estimate
            "bias": bias,                                     # bias
            "rel_bias_pct": relative_bias_pct,                # relative bias
            "std": std,                                       # estimate sd
            "se_mean": se_mean,                               # SE of mean
            "ci95_lo": ci95_lo,                               # lower CI
            "ci95_hi": ci95_hi,                               # upper CI
            "mean_abs_bias": mean_abs_bias,                   # mean abs bias
            "rmse": rmse,                                     # RMSE
            "conv_rate": g["converged"].mean(),               # convergence rate
            "mean_success_starts": g["n_success_starts"].mean(), # avg starts
            "median_local_gap": g["local_gap"].median(),      # median gap
        })

    summary = pd.DataFrame(summary_rows)                       # summary table
    return df, summary                                         # long and summary


In [ ]:
# Run Monte Carlo and collect long-form and summary results.
results, summary = run_monte_carlo()                           # MC outputs


## 11. Results

In [ ]:
print("=" * 75)                                                  # top divider
print("NFXP Monte Carlo — original finite-type model")            # title
print(f"K={K}, J={J}, T={T}, N={N}, D_MAX={D_MAX}, reps={MC_REPS}") # setup
print("=" * 75)                                                  # bottom divider
print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}")) # summary


## 12. Visualization

Boxplots for all 6 parameters over Monte Carlo replications compared to the true values.

In [ ]:
# LaTeX labels for each parameter used in plots.
latex_lbl = {                                                    # plot labels
    "alpha_2_1": r"$\alpha_2^1$",                              # type 1 alpha
    "alpha_2_2": r"$\alpha_2^2$",                              # type 2 alpha
    "pi_1": r"$\pi_1$",                                        # type 1 share
    "gamma": r"$\gamma$",                                      # price coefficient
    "beta_sc": r"$\beta^{sc}$",                                # switching cost
    "beta_dep": r"$\beta^{dep}$",                              # depletion rate
}

fig, axes = plt.subplots(2, 3, figsize=(12, 7))                  # subplot grid
axes = axes.flatten()                                           # 1D axis list

for ax, (i, name) in zip(axes, enumerate(PARAM_NAMES)):         # parameter panels
    est = results[results["param"] == name]["estimate"].to_numpy() # estimates
    true_v = THETA_TRUE[i]                                      # true value
    ax.boxplot(est, vert=True, patch_artist=True,               # estimate boxplot
               boxprops=dict(facecolor="steelblue", alpha=0.7), # box style
               medianprops=dict(color="navy", linewidth=2))    # median style
    ax.axhline(true_v, color="firebrick", lw=1.8, ls="--",      # true line
               label=f"True = {true_v:.3f}")                   # true label
    ax.set_title(latex_lbl.get(name, name), fontsize=12)        # panel title
    ax.set_ylabel("Estimate", fontsize=9)                       # y label
    ax.legend(fontsize=8)                                       # legend
    ax.xaxis.set_major_locator(mticker.NullLocator())           # hide x ticks

fig.suptitle(                                                   # figure title
    r"NFXP Monte Carlo: original finite-type model  ($K=2$)"
    f"  (reps={MC_REPS}, N={N}, T={T})",
    fontsize=11,                                                # title size
)
plt.tight_layout()                                              # compact layout
plt.savefig("nfxp_mc_boxplots_original_finite_types.pdf", bbox_inches="tight") # save
plt.show()                                                      # display


In [ ]:
# Bias and RMSE bar chart summarizing estimation accuracy.
labels = [latex_lbl.get(n, n) for n in PARAM_NAMES]             # x labels
bias_abs = [abs(summary.loc[summary["param"] == n, "bias"].values[0]) for n in PARAM_NAMES] # abs bias
rmse_v = [summary.loc[summary["param"] == n, "rmse"].values[0] for n in PARAM_NAMES] # RMSE

x = np.arange(len(PARAM_NAMES))                                 # x positions
w = 0.35                                                        # bar width

fig, axes = plt.subplots(1, 2, figsize=(12, 4))                 # figure grid

for ax, vals, ylabel, title in [                                # metric panels
    (axes[0], rmse_v, "RMSE", "RMSE per parameter"),           # RMSE panel
    (axes[1], bias_abs, "|Bias|", "|Bias| per parameter"),      # bias panel
]:
    ax.bar(x, vals, 2 * w, color="steelblue", alpha=0.85, edgecolor="black", lw=0.5) # bars
    ax.set_xticks(x)                                            # tick positions
    ax.set_xticklabels(labels, fontsize=10)                     # tick labels
    ax.set_ylabel(ylabel)                                       # y label
    ax.set_title(title)                                         # panel title
    ax.axhline(0, color="black", lw=0.5)                        # zero line

fig.suptitle(                                                   # figure title
    r"NFXP Monte Carlo — finite-type error metrics  ($K=2$)"
    f"  (reps={MC_REPS}, N={N}, T={T})",
    fontsize=10,                                                # title size
)
plt.tight_layout()                                              # compact layout
plt.savefig("nfxp_mc_bias_rmse_original_finite_types.pdf", bbox_inches="tight") # save
plt.show()                                                      # display
